- 3a Stochastic pulse distribution
- 3b Noise floor
- 3c Trigger and acquisition window
- 3d Charge integration
- 3e Neutron/gamma channels (3D plot)

In [ ]:
# Imports
from typing import Callable, Literal
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable
import mpl_toolkits.mplot3d.art3d as art3d
import numpy as np
from scipy.optimize import curve_fit
import pandas as pd
from data_processing import processing as proc
from data_processing import loading as load
from data_processing import types as proc_types
from data_processing.dataframe_validation import DetectorDataframeColumn, EnergyColumn, get_df_col
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import (
    get_input_with_default,
    stop
)

### Functions

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

In [ ]:
# Functions
pileup_flag = 0x8000


def is_pileup_flag(flags: int) -> bool:
    return flags & pileup_flag != 0

In [ ]:
def get_nasa_generation_settings(
    calib_key: CalibrationKey
) -> proc_types.NasaGenerationSettings:
    sigma = get_input_with_default(
        """\
Enter value of sigma
Press Enter for default (5)
""",
        5,
        float
    )
    window_offset = get_input_with_default(
        """\
Enter value of offset between top and bottom window border
Press Enter for default (0.2)
""",
        0.2,
        float
    )
    left_border_type_input = get_input_with_default(
        """\
How do you want to handle the left border?
1: use existing value (default)
2: recalculate from data
3: enter own value
Press Enter for default
""",
        1,
        int
    )
    if left_border_type_input == 1:
        existing_left_border_version_input = get_input_with_default(
            """\
Which existing left border do you want to use?
1: original (0.1966 MeVee) (default)
2: newer (~0.1866 MeVee)
or press Enter for default
""",
            1,
            int
        )
        if existing_left_border_version_input in [1, 2]:
            border_key = ExperimentDataKey.NASA_BORDERS if existing_left_border_version_input == 1 else ExperimentDataKey.NASA_BORDERS_RECALC
            file_name_prefix = f"{calib_key.value}_{border_key.value}"
            side_borders_path, *_ = load.get_neutron_window_paths(file_name_prefix=file_name_prefix)
            left_border, _ = load.load_side_borders(side_borders_path=side_borders_path)
            if left_border is None:
                raise ValueError("Left border could not be loaded")
            lower_energy_bound = left_border
            recalc_lower_bound = False
        else:
            raise ValueError("Unsupported choice")
        pass
    elif left_border_type_input == 2:
        lower_energy_bound = 0.1966
        recalc_lower_bound = True
    elif left_border_type_input == 3:
        lower_energy_bound = get_input_with_default(
            """\
Enter value of lower energy bound (in MeVee)
Press Enter for default (0.1966)
""",
            0.1966,
            float
        )
        recalc_lower_bound = False
    else:
        raise ValueError("Unsupported choice")
    settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
    return settings


def make_strategy_factory_fn(
    strategy_factory: proc.NeutronStrategyFactory,
    window_type: proc_types.WindowType,
    loading: bool,
    settings: proc_types.NeutronWindowSettings
) -> Callable[[], proc.AbstractNeutronStrategy]:
    def factory_fn():
        return strategy_factory.make_neutron_window_strategy(
            window_type, loading, settings
        )
    return factory_fn


def make_strategy_for_experiments(
    experiment_neutron_data: ExperimentNeutronData, 
    factory_fn: Callable[[], proc.AbstractNeutronStrategy]
) -> ExperimentNeutronData:
    new_neutron_data = {
        exp_id: {**exp_data, ExperimentDataKey.BORDER_STRATEGY: factory_fn()}
        for exp_id, exp_data
        in experiment_neutron_data.items()
    }
    return new_neutron_data

In [ ]:
def moving_average(arr, n=5):
    ret = np.cumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((n-1,))
    prefix[:] = np.nan
    return np.concatenate((prefix, mov_avg))


def moving_average_centered(arr, n=5):
    if n % 2 != 1:
        raise ValueError("Centered moving average needs odd window size")
    prefix_count = (n-1)//2
    ret = np.nancumsum(arr, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    mov_avg = ret[n-1:] / n
    prefix = np.empty((prefix_count,))
    suffix = np.empty((prefix_count,))
    prefix[:] = np.nan
    suffix[:] = np.nan
    return np.concatenate((prefix, mov_avg, suffix))

In [ ]:
def correct_raw_signals(
    raw_signals_df: pd.DataFrame,
    baseline_idx_range: int = 30,
    max_adc: int = 16367,
    baseline_offset: float = 0.10
) -> pd.DataFrame:
    signals_np = raw_signals_df.to_numpy()
    signals_np = -signals_np + max_adc
    corrected_signals = pd.DataFrame(
        signals_np,
        index=raw_signals_df.index,
        columns=raw_signals_df.columns
    )
    return corrected_signals

In [ ]:
def textbox(text, x, y, width, height, facecolor, textcolor, ax, text_x_offset=0.5, text_y_offset=0.5, ha="center"):
    rect_params = {
        "linewidth": 0,
        # "ec": "black",
        "fc": facecolor
    }
    text_params = {
        "ha": ha,
        "va": "center",
        "fontsize": fontsize,
        "color": textcolor
    }
    rect = mpl.patches.Rectangle((x, y), width, height, **rect_params)
    ax.add_patch(rect)
    ax.annotate(
        # (t_2 + t_3) / 2,
        text,
        (text_x_offset, text_y_offset),
        xycoords=rect,
        **text_params
    )

In [ ]:
def get_bbox_center(bbox: mpl.transforms.BboxBase) -> tuple[float, float]:
    return (bbox.xmin + bbox.width / 2, bbox.ymin + bbox.height / 2)


def get_translation_to(
    pos_from: tuple[float, float],
    pos_to: tuple[float, float]
) -> tuple[float, float]:
    x_from, y_from = pos_from
    x_to, y_to = pos_to
    return (x_to - x_from, y_to - y_from)


def make_text_path(
    text: str | list[str],
    linespacing: float,
    font_properties=None,
    usetex=False
) -> mpl.text.TextPath:
    paths = []
    
    if isinstance(text, str):
        text = [text]
    
    for i, line in enumerate(text):
        text_path = mpl.text.TextPath(
            (0, 0), line,
            prop=font_properties,
            size=1,
            usetex=usetex
        )
        bbox = text_path.get_extents()
        line_width = bbox.width
        from_x = bbox.xmin
        from_y = bbox.xmax
        # center first line on x=0, anchor at y=0, next lines below
        to_x = -line_width / 2
        to_y = -i * linespacing
        x = to_x - from_x
        y = to_y - from_y
        
        line_transform = mpl.transforms.Affine2D().translate(x, y)
        transformed_path = line_transform.transform_path(text_path)
        paths.append(transformed_path)
    
    text_path = mpl.path.Path.make_compound_path(*paths)
    return text_path


def make_bounding_box_for_text_path(
    text_path: mpl.text.TextPath,
    padding: float,
    rounding_size: float,
    zorder: int,
    mutation_aspect: float = 1,
    **bbox_params
) -> mpl.patches.FancyBboxPatch:
    text_bbox = text_path.get_extents()
    # get anchor corner, width, height
    # make FancyBboxPatch
    anchor = text_bbox.min
    boxstyle = f"round, pad={padding}, rounding_size={rounding_size}"
    return mpl.patches.FancyBboxPatch(
        anchor,
        text_bbox.width,
        text_bbox.height,
        boxstyle=boxstyle,
        mutation_aspect=mutation_aspect,
        **bbox_params
    )


def convert_text_path_to_patch(
    text_path: mpl.text.TextPath,
    zorder: int
) -> mpl.patches.PathPatch:
    return mpl.patches.PathPatch(text_path, ec="none", fc="k", zorder=zorder)


def patch_to_3d_plot_wall(
    patch: mpl.patches.Patch,
    ax: mpl.axes.Axes,
    zdir: str,
    z: float = 0,
):
    ax.add_patch(patch)
    art3d.pathpatch_2d_to_3d(patch, z=z, zdir=zdir)


def get_center_match_transform(
    box_from: mpl.transforms.Bbox,
    box_to: mpl.transforms.Bbox
) -> tuple[float, float]:
    from_c_x = (box_from.x0 + box_from.x1) / 2
    from_c_y = (box_from.y0 + box_from.y1) / 2
    to_c_x = (box_to.x0 + box_to.x1) / 2
    to_c_y = (box_to.y0 + box_to.y1) / 2
    return (to_c_x - from_c_x, to_c_y - from_c_y)

In [ ]:
def get_psd_adc_histogram(
    df: pd.DataFrame,
    adc_width: float = 420,
    adc_bins: np.ndarray | None = None,
    psd_bin_count: int = 100,
    psd_min: float = 0.0,
    psd_max: float = 0.5
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    x = get_df_col(df, DetectorDataframeColumn.ENERGY)
    y = get_df_col(df, DetectorDataframeColumn.PSD)

    within_psd = y.between(psd_min, psd_max)
    x = x[within_psd == True].copy()
    y = y[within_psd == True].copy()

    if adc_bins is not None:
        x_bins = adc_bins
    else:
        x_bins: np.ndarray = np.linspace(
            0, x.max(), int(x.max() / adc_width) + 1
        )
    print(f"Energy width = {x_bins[1]-x_bins[0]} ADC")
    y_bins: np.ndarray = np.linspace(psd_min, psd_max, psd_bin_count + 1)

    Z, xe, ye = np.histogram2d(x, y, bins=[x_bins, y_bins])
    return Z, xe, ye

## Experiment ID Input

In [ ]:
experiment_ids = ["test_noise_floor_26.04.24", "TB-26"]

In [ ]:
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

In [ ]:
calib_input = "y"

is_new_calibration = calib_input.lower() == "y"
calibrated_energy_column: EnergyColumn = (
    DetectorDataframeColumn.RECALIBRATED_ENERGY
    if is_new_calibration else DetectorDataframeColumn.CALIB_ENERGY
)
calib_key: CalibrationKey = ExperimentDataKey.NEW_CALIBRATION if is_new_calibration else ExperimentDataKey.CAEN_CALIBRATION

In [ ]:
strategy_factory = proc.NeutronStrategyFactory()
window_offset = 0.2
sigma = 5
lower_energy_bound = 0.05
recalc_lower_bound = False
settings = proc_types.NasaGenerationSettings(
        window_offset=window_offset,
        sigma=sigma,
        lower_energy_bound=lower_energy_bound,
        recalculate_lower_energy_bound=recalc_lower_bound
    )
factory_fn = make_strategy_factory_fn(
    strategy_factory, "nasa", False, settings)
experiment_neutron_data = make_strategy_for_experiments(
    experiment_neutron_data, factory_fn)

In [ ]:
unconv_folder = Path("Q:/Neutron Data/1-Unconverted_Data")
long_window_dataset_name = "TB-long_window"
exp_folder = unconv_folder / long_window_dataset_name

In [ ]:
raw_folder = exp_folder / "RAW"
data_file = raw_folder / "SDataR_TB-long_window.CSV"

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    if exp_id == "TB-26":
        exp_data[ExperimentDataKey.UNCLASSIFIED] = load.load_parquet_psd(exp_id, with_flags=True)
    exp_data["signals_df"] = load.load_parquet_signals(exp_id)

In [ ]:
headers = []
samples_lines = []
samples_start_idx = 0
scan_start = 1
scan_len = 1000
scan_end = scan_start + scan_len
last_idx = 0

with open(data_file) as fp:
    for i, line in enumerate(fp):
        if i == 0:
            headers = line.strip().split(";")
            samples_start_idx = headers.index("SAMPLES")
        elif scan_start <= i < scan_end:
            possible_line = line.strip().split(";")[samples_start_idx:]
            samples_lines.append(possible_line)
            last_idx = i
        elif i >= scan_end:
            break
samples = np.array(samples_lines, dtype=int)
print(samples.shape)
exp_data = {"samples": samples}
experiment_neutron_data[long_window_dataset_name] = exp_data

In [ ]:
# Express timetags in hours elapsed
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data.get(ExperimentDataKey.UNCLASSIFIED)
    if unclassified_df is not None:
        unclassified_df = load.calculate_timetag_hours(unclassified_df)
        exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

In [ ]:
# Recalibrate energy
for exp_id, exp_data in experiment_neutron_data.items():
    unclassified_df = exp_data.get(ExperimentDataKey.UNCLASSIFIED)
    if unclassified_df is not None:
        unclassified_df = proc.recalibrate(unclassified_df, proc.Detector.ZERO)
        exp_data[ExperimentDataKey.UNCLASSIFIED] = unclassified_df

## Data Processing

### Figure 3b Processing

In [ ]:
exp_id = "test_noise_floor_26.04.24"

In [ ]:
exp_data = experiment_neutron_data[exp_id]
signals_df = exp_data["signals_df"]
signals_df = signals_df.astype("int32")
signals_np = signals_df.to_numpy()
baselines = signals_np[:, :30].mean(axis=1).reshape(-1, 1)
signals_np = -signals_np + baselines
signals_df.columns = signals_df.columns.map(int)
signals_df = pd.DataFrame(signals_np, index=signals_df.index, columns=signals_df.columns)
exp_data["signals_df"] = signals_df

In [ ]:
exp_data = experiment_neutron_data[exp_id]
signals_df = exp_data["signals_df"]
histo_data = signals_df.iloc[:, :40]
histo_times = np.tile(
    histo_data.columns.map(lambda x: x * 2),
    histo_data.shape[0]
)
histo_es = np.ravel(histo_data.to_numpy(), order="F")

time_bins = np.arange(0, 82, 2)
e_min = -500
e_max = 750
e_bins = np.linspace(e_min, e_max, int((e_max - e_min) / 10)+1)
histo_results = np.histogram2d(histo_times, histo_es, bins=(time_bins, e_bins))
histo_counts, histo_time_bins, histo_e_bins = histo_results
collapsed_histo_energy = histo_counts.sum(axis=0)
exp_data["histo_results"] = {
    "counts": histo_counts,
    "time_bins": histo_time_bins,
    "e_bins": histo_e_bins,
    "collapsed_histo_energy": collapsed_histo_energy
}

In [ ]:
exp_data = experiment_neutron_data[exp_id]
histo_results = exp_data["histo_results"]
collapsed_histo = histo_results["collapsed_histo_energy"]
e_bins = histo_results["e_bins"]
e_mids = (e_bins[1:] + e_bins[:-1]) / 2

fit_params, _ = curve_fit(proc.gaussian, e_mids, collapsed_histo)
print(fit_params)
histo_results["fit_params"] = fit_params

### Figure 3c/d Processing

#### Neutron Classification

In [ ]:
exp_id = "TB-26"

In [ ]:
# Generate histogram

start_scan_idx = 0
end_scan_idx = 420
energy_width = 20e-3

exp_data = experiment_neutron_data[exp_id]
psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
Z, xe, ye = proc.get_psd_energy_histogram(
    psd_report,
    calibrated_energy_column,
    energy_width=energy_width
)
exp_data[ExperimentDataKey.PSD_HISTOGRAM] = Z
exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES] = xe
exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES] = ye
exp_data[ExperimentDataKey.END_SCAN_IDX] = min(end_scan_idx, len(Z))

In [ ]:
exp_data = experiment_neutron_data[exp_id]
stop_here = False

exp_data = experiment_neutron_data[exp_id]
Z = exp_data[ExperimentDataKey.PSD_HISTOGRAM]
xe = exp_data[ExperimentDataKey.HISTOGRAM_X_EDGES]
ye = exp_data[ExperimentDataKey.HISTOGRAM_Y_EDGES]
end_scan_idx = exp_data[ExperimentDataKey.END_SCAN_IDX]

df, df_err = proc.scan_histogram_slices(
    Z,
    xe,
    ye,
    fit_style="peak_finder",
    start_idx=start_scan_idx,
    end_idx=end_scan_idx
)
df, bad_slice_indexes = proc.find_failed_slices(df, exp_id)

if bad_slice_indexes is not None:
    exp_data[ExperimentDataKey.VALID_SLICE_FITS] = df
    exp_data[ExperimentDataKey.BAD_SLICE_INDEXES] = bad_slice_indexes
    stop_here = True
else:
    # exp_data['fom_results'] = df
    exp_data[ExperimentDataKey.FOM_RESULTS] = df

if stop_here:
    stop()

In [ ]:
# get borders from strategy
exp_data = experiment_neutron_data[exp_id]
if ExperimentDataKey.FOM_RESULTS not in exp_data:
    print(f"No good fit data on Experiment {exp_id}")
else:
    fom_results = exp_data[ExperimentDataKey.FOM_RESULTS]
    strategy = exp_data[ExperimentDataKey.BORDER_STRATEGY]
    
    strategy.set_slice_fit_dataframe(fom_results)
    borders = strategy.get_neutron_window()
    
    exp_data[ExperimentDataKey.BORDERS] = borders

In [ ]:
# classify neutrons
exp_data = experiment_neutron_data[exp_id]
psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED].copy()
borders = exp_data[ExperimentDataKey.BORDERS]

psd_report = proc.classify(
    psd_report,
    calibrated_energy_column,
    borders,
    DetectorDataframeColumn.NEW_N_CLASS
)

exp_data[ExperimentDataKey.PSD_REPORT] = psd_report

### Pulse Selection

In [ ]:
exp_data = experiment_neutron_data[exp_id]
psd_report = exp_data[ExperimentDataKey.PSD_REPORT]
n_class_col_name = DetectorDataframeColumn.NEW_N_CLASS.value

gamma_only = psd_report.query(f"~{n_class_col_name}").copy()
neutrons_only = psd_report.query(n_class_col_name).copy()
exp_data[ExperimentDataKey.NEUTRONS_ONLY] = neutrons_only
exp_data[ExperimentDataKey.GAMMA_ONLY] = gamma_only

In [ ]:
exp_data = experiment_neutron_data[exp_id]
neutrons_only = exp_data[ExperimentDataKey.NEUTRONS_ONLY]
gamma_only = exp_data[ExperimentDataKey.GAMMA_ONLY]
signals_df = exp_data["signals_df"]

neutron_signals = signals_df.loc[neutrons_only.index].astype("int32")
gamma_signals = signals_df.loc[gamma_only.index].astype("int32")

neutron_signals = correct_raw_signals(neutron_signals)
gamma_signals = correct_raw_signals(gamma_signals)

exp_data["neutron_signals"] = neutron_signals
exp_data["gamma_signals"] = gamma_signals

In [ ]:
min_height = 13000
max_height = 14500

exp_data = experiment_neutron_data[exp_id]
neutron_signals = exp_data["neutron_signals"]
gamma_signals = exp_data["gamma_signals"]

selected_neutron = None
selected_gamma = None

bad_neutrons = [64413, 67314, 125305, 127752]
bad_gamma = []

for neutron_id, neutron_signal in neutron_signals.iterrows():
    # get clean neutron pulse (no secondary peak)
    if selected_neutron is not None:
        break
    if neutron_id in bad_neutrons:
        continue
    n_height = neutron_signal.max()
    if n_height < min_height or n_height > max_height:
        continue
    else:
        # get matching clean gamma pulse
        for gamma_id, gamma_signal in gamma_signals.iterrows():
            if selected_gamma is not None:
                break
            if gamma_id in bad_gamma:
                continue
            g_height = gamma_signal.max()
            if abs(g_height - n_height) > (0.01 * n_height):
                continue
            print(f"Neutron ID = {neutron_id}, height = {n_height}")
            print(f"Gamma ID = {gamma_id}, height = {g_height}")
            selected_neutron = neutron_id, neutron_signal
            selected_gamma = gamma_id, gamma_signal

if selected_neutron is None or selected_gamma is None:
    raise Exception("No pulse found")
exp_data["selected_neutron"] = selected_neutron
exp_data["selected_gamma"] = selected_gamma

### Figure 3e Processing

In [ ]:
exp_id = "TB-26"

In [ ]:
# Generate histogram
adc_width = 20

exp_data = experiment_neutron_data[exp_id]
psd_report = exp_data[ExperimentDataKey.UNCLASSIFIED]
Z, xe, ye = get_psd_adc_histogram(
    psd_report,
    adc_width=adc_width
)
adc_histogram = {
    "histogram": Z,
    "x_edges": xe,
    "y_edges": ye,
}
exp_data["adc_histogram"] = adc_histogram

## Figure Base Data

In [ ]:
dl_folder = Path.home() / "Downloads" / "neutron_detection_paper"
dl_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_figure_3a_data() -> pd.DataFrame:
    """
    Each series is one sample from the neutron detector
    Each value in the series is the light level as reported by the detector ADC
    (The value starts high for no light, and drops with increasing intensity)
    The index is the time of each ADC value recording in us
    (The first 5 us are omitted)
    """
    figa_data = experiment_neutron_data[long_window_dataset_name]
    samples = figa_data["samples"]
    x_data = np.arange(0, samples.shape[1] - 5000) * 2 / 1000
    df = pd.DataFrame(samples[:, 5000:].T, index=x_data)
    return df

In [ ]:
def get_figure_3b_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    First DataFrame contains noise samples from the neutron detector.
    Each series is one value in a sample data set
    The first row is the time for each sample value in ns
    The following rows are the light levels for each sample
    Second DataFrame contains histogram of noise sample values
    (This uses actual values over time, not averages for each sample)
    Index is light level, series values are counts at that light level
    """
    exp_data = experiment_neutron_data["test_noise_floor_26.04.24"]
    signals_df = exp_data["signals_df"]
    histo_results = exp_data["histo_results"]
    e_bins = histo_results["e_bins"]
    collapsed_histo_energy = histo_results["collapsed_histo_energy"]

    e_mids = (e_bins[1:] + e_bins[:-1]) / 2

    pulse_x = signals_df.columns.map(lambda x: int(x) * 2)
    pulse_x_subset = pulse_x[:40]
    pulse_x_series = pd.Series(pulse_x_subset)
    subset_signals_df = signals_df.iloc[:, :40]
    new_index = subset_signals_df.index.map(lambda x: f"Sample {x}")
    subset_signals_df = subset_signals_df.set_index(new_index)
    subset_signals_df = subset_signals_df.T
    subset_signals_df.insert(0, "Time (ns)", pulse_x_series)
    subset_signals_df = subset_signals_df.T

    histo_df = pd.DataFrame(data={"counts": collapsed_histo_energy}, index=e_mids)

    return subset_signals_df, histo_df

In [ ]:
def get_figure_3c_data() -> pd.DataFrame:
    """
    Contains data from a selected neutron pulse
    Index is time in ns, series values are neutron pulse heights in ADC channels
    """
    exp_data = experiment_neutron_data["TB-26"]
    _, selected_neutron = exp_data["selected_neutron"]

    neutron_y = selected_neutron.values
    neutron_y = np.pad(neutron_y, 50, mode="edge")
    neutron_x = np.arange(-50, len(neutron_y) - 50) * 2
    df = pd.DataFrame(data={"Pulse height (ADC channels)": neutron_y}, index=neutron_x)
    return df

In [ ]:
def get_figure_3d_data() -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    First DataFrame contains data from a selected neutron pulse
    Second DataFrame contains data from a selected gamma pulse
    In both cases, index is time in ns, series values are pulse heights in ADC channels
    """
    exp_data = experiment_neutron_data["TB-26"]
    _, selected_neutron = exp_data["selected_neutron"]
    _, selected_gamma = exp_data["selected_gamma"]

    neutron_y = selected_neutron.values
    gamma_y = selected_gamma.values
    neutron_x = np.arange(0, len(selected_neutron)) * 2
    gamma_x = np.arange(0, len(selected_gamma)) * 2
    neutron_df = pd.DataFrame(data={"y": neutron_y}, index=neutron_x)
    gamma_df = pd.DataFrame(data={"y": gamma_y}, index=gamma_x)
    return neutron_df, gamma_df

In [ ]:
def get_figure_3e_data(
    e_margin: float = 2,
    psd_margin: float = 0.002
) -> pd.DataFrame:
    """
    Contains 2D histogram of neutron detector data
    Each row represents one bar in the histogram
    The columns are the parameters needed to place that bar in the 3D plot
    """
    exp_data = experiment_neutron_data["TB-26"]
    histo_dict = exp_data["adc_histogram"]
    xe = histo_dict["x_edges"]
    ye = histo_dict["y_edges"]
    dz = histo_dict["histogram"]

    x, y = np.meshgrid(xe[:-1], ye[:-1])
    x, y = x.ravel(), y.ravel()
    z = np.full_like(x, 0)
    _dx = xe[1:] - xe[:-1]
    _dy = ye[1:] - ye[:-1]
    dx, dy = np.meshgrid(_dx, _dy)
    dz = dz.T
    dx, dy, dz = dx.ravel(), dy.ravel(), dz.ravel()

    x = x + e_margin
    dx = dx - e_margin
    y = y + psd_margin
    dy = dy - psd_margin

    height_mask = dz > 20
    x = x[height_mask]
    y = y[height_mask]
    z = z[height_mask]
    dx = dx[height_mask]
    dy = dy[height_mask]
    dz = dz[height_mask]

    df = pd.DataFrame(data={"x": x, "y": y, "z": z, "dx": dx, "dy": dy, "dz": dz})
    return df

## Plotting

### Plot Style Constants

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']

fontsize = 7
fontsize_small = 5
linewidth = 1
plt.rcParams['font.size'] = fontsize
plt.rcParams['lines.linewidth'] = linewidth

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"

### Plot Functions

#### Plot Helpers

In [ ]:
def get_heights_for_boxes(n_boxes: int, rel_height: float = 1.2, margin_pts: float = 1):
    font_height_inches = fontsize / 72  # 72 pts in 1 inch
    margin_height_units = margin_pts / 72
    box_height_units = rel_height * font_height_inches
    
    axes_height = n_boxes*box_height_units + (n_boxes-1)*margin_height_units
    box_height_relative = box_height_units / axes_height
    margin_relative = margin_height_units / axes_height

    box_y_coords = [i*(box_height_relative+margin_relative) for i in range(n_boxes)]
    
    return box_y_coords, axes_height, box_height_relative

In [ ]:
def set_3d_ticklabel_alignment(ax: mpl.axes.Axes):
    xaxis_ticklabels = ax.xaxis.get_ticklabels()
    for ticklabel in xaxis_ticklabels:
        ticklabel.set_ha("right")
        ticklabel.set_va("baseline")
    yaxis_ticklabels = ax.yaxis.get_ticklabels()
    for ticklabel in yaxis_ticklabels:
        ticklabel.set_ha("center")
        ticklabel.set_va("top")
    zaxis_ticklabels = ax.zaxis.get_ticklabels()
    for ticklabel in zaxis_ticklabels:
        ticklabel.set_ha("left")
        ticklabel.set_va("center_baseline")

In [ ]:
def set_3d_ticklabel_padding(
    ax: mpl.axes.Axes, x_pad: float, y_pad: float, z_pad: float
):
    # all padding is relative to fontsize
    ax.tick_params(axis="x", pad=x_pad*fontsize)
    ax.tick_params(axis="y", pad=y_pad*fontsize)
    ax.tick_params(axis="z", pad=z_pad*fontsize)

In [ ]:
def set_3d_axis_label_padding(
    ax: mpl.axes.Axes, x_pad: float, y_pad: float, z_pad: float
):
    ax.xaxis.labelpad = x_pad * fontsize
    ax.yaxis.labelpad = y_pad * fontsize
    ax.zaxis.labelpad = z_pad * fontsize

In [ ]:
def set_3d_panes_transparent(ax: mpl.axes.Axes):
    ax.xaxis.set_pane_color((1, 1, 1, 0))
    ax.yaxis.set_pane_color((1, 1, 1, 0))
    ax.zaxis.set_pane_color((1, 1, 1, 0))

#### Main Functions

In [ ]:
def plot_figure_3a(ax: mpl.axes.Axes):
    fig_3a_data = get_figure_3a_data()

    for _, series in fig_3a_data.items():
        ax.plot(
            series.index,
            series,
            color="black",
            alpha=0.5
        )

    ax.set_xlim(-1, 31)
    ax.grid(alpha=0.5)
    ax.set_xlabel(r"Time ($\mu$s)")
    ax.set_ylabel("Pulse height (ADC channel)")
    ax.tick_params(labelsize=fontsize_small)

In [ ]:
def plot_figure_3b(ax: mpl.axes.Axes):

    series_df, histo_df = get_figure_3b_data()

    signals_df, _ = get_figure_3b_data()
    pulse_time = signals_df.loc["Time (ns)"]
    samples_mask = ["Sample" in x for x in signals_df.index]
    samples_df = signals_df.loc[samples_mask]
    samples_np = samples_df.to_numpy()

    divider = make_axes_locatable(ax)
    ax_h = divider.append_axes("right", "40%", pad=0, sharey=ax)

    for pulse_values in samples_np:
        ax.plot(pulse_time, pulse_values,
                color=bg_blue,
                alpha=0.2)
    ax_h.fill_betweenx(histo_df.index, 0, histo_df["counts"], color=bg_blue)

    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("Pulse height (ADC channel)")
    ax_h.set_xlabel("Counts")

    ax.set_xlim(0, 80)

    ax.tick_params(labelsize=fontsize_small)
    ax_h.tick_params(labelsize=fontsize_small)
    ax_h.set_xscale("log")
    ax_h.xaxis.set_major_locator(mpl.ticker.LogLocator(numticks=5))
    ax_h.yaxis.set_tick_params(labelleft=False)

In [ ]:
def plot_figure_3c(ax: mpl.axes.Axes):
    t_t = 144
    pulse_time = 400
    max_adc = 16367

    neutron_df = get_figure_3c_data()

    box_y_coords, axes_height, box_height_relative = get_heights_for_boxes(1)

    # Additional axes
    divider = make_axes_locatable(ax)
    ax2 = divider.append_axes(
        "bottom",
        axes_height,
        pad=3*axes_height,
        sharex=ax
    )

    ax.plot(
        neutron_df.index,
        neutron_df["Pulse height (ADC channels)"],
        label="Neutron"
    )

    # Time interval textboxes on axis2
    box_y = box_y_coords[0]
    textbox("Pre-trigger", 0, box_y, t_t, box_height_relative, bg_grey, "white", ax2)

    # Connecting line
    trigger_line_height = 2800
    con = mpl.patches.ConnectionPatch(
        (t_t, 1),
        (t_t, trigger_line_height),
        "data",
        "data",
        axesA=ax2,
        axesB=ax,
        linestyle=":",
        color=bg_red
    )
    ax2.add_artist(con)

    # CFD trigger annotation
    ax.text(
        t_t+2,
        trigger_line_height - 275,
        "CFD Trigger",
        ha="left",
        va="center",
        fontsize=fontsize_small
    )

    # Acquisition window box
    ax.text(
        pulse_time / 2,
        max_adc,
        "Acquisition Window",
        ha="center",
        va="bottom",
        fontsize=fontsize_small
    )
    box_width_offset = 2
    box_height_offset = 50
    box = mpl.patches.Rectangle(
        (box_width_offset, box_height_offset),
        pulse_time-2*box_width_offset,
        max_adc-2*box_height_offset,
        ec="black",
        fc="#f5f5f5",
        zorder=0,
    )
    ax.add_patch(box)

    ax.set_ylabel("Pulse height (ADC channel)")
    ax.set_xlabel("Time (ns)")

    ax.set_ylim(0, 17000)
    ax.set_xlim(-100, pulse_time+100)

    ax.xaxis.set_major_formatter(
        lambda x, _: str(int(x)) if x >= 0 and x <= 400 else ""
    )
    ax.yaxis.set_major_formatter(mpl.ticker.NullFormatter())
    ax.tick_params(axis="y", labelleft=False)
    ax.tick_params(labelsize=fontsize_small)
    for tick in ax.xaxis.get_majorticklabels():
        tick_x, tick_y = tick.get_position()
        if tick_x == 150.0:
            tick.set_ha("left")
    ax.spines["bottom"].set_position("zero")
    ax.spines["left"].set_position("zero")
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)

    ax2.patch.set_visible(False)
    ax2.axis("off")

In [ ]:
def plot_figure_3d(ax: mpl.axes.Axes):
    t_t = 144
    pulse_time = 400
    pre_gate = 50
    short_gate = 22
    gate_width = 250

    t_1 = t_t - pre_gate
    t_2 = t_1 + short_gate
    t_3 = t_1 + gate_width
    
    neutron_df, gamma_df = get_figure_3d_data()
    neutron_x = neutron_df.index
    neutron_y = neutron_df["y"]
    gamma_x = gamma_df.index
    gamma_y = gamma_df["y"]

    textbox_ys, axes_height, box_height_relative = get_heights_for_boxes(3)

    # Additional axes
    divider = make_axes_locatable(ax)
    ax2 = divider.append_axes(
        "bottom",
        axes_height,
        pad=1.25*axes_height,
        sharex=ax
    )
    sec = ax.secondary_xaxis(location=0)

    ax.plot(neutron_x, neutron_y, label="Neutron")
    ax.plot(gamma_x, gamma_y, label="Gamma")
    ax.fill_between(
        neutron_x, neutron_y, gamma_y, where=(100 <= neutron_x), color=bg_blue, interpolate=True
    )

    # Time interval textboxes on axis2
    textbox_labels = ["Short gate (head integral)", "Gate (total integral)", "Pre-gate"]
    textbox_xs = [t_1, t_1, t_1]
    textbox_widths = [short_gate, gate_width, pre_gate]
    textbox_text_colors = ["black", "white", "white"]
    textbox_kwargs = [{"ha": "left", "text_x_offset": 1.1}, {}, {}]
    textbox_params_zip = zip(
        textbox_labels, textbox_xs, textbox_ys, textbox_widths,
        textbox_text_colors, textbox_kwargs
    )
    for i, params in enumerate(textbox_params_zip):
        label, x, y, width, text_color, kwargs = params
        textbox(label, x, y, width, box_height_relative, bg_grey, text_color, ax2, **kwargs)

    # Connecting line
    trigger_line_height = 2800
    con = mpl.patches.ConnectionPatch(
        (t_t, textbox_ys[-1]),
        (t_t, trigger_line_height),
        "data",
        "data",
        axesA=ax2,
        axesB=ax,
        linestyle=":",
        color=bg_red
    )
    ax2.add_artist(con)

    # CFD trigger annotation
    ax.text(
        t_t + 2,
        trigger_line_height - 275,
        "CFD Trigger",
        ha="left",
        va="center",
        fontsize=fontsize_small
    )

    ax.set_ylabel("Pulse height (ADC channel)")
    sec.set_xlabel("Time (ns)")

    ax.set_ylim(0, 17000)
    ax.set_xlim(0, pulse_time)

    ax.tick_params(labelsize=fontsize_small)
    for tick in ax.xaxis.get_majorticklabels():
        tick_x, tick_y = tick.get_position()
        if tick_x == 150.0:
            tick.set_ha("left")
    ax.yaxis.set_major_formatter(mpl.ticker.NullFormatter())
    sec.set_xticks([t_1, t_2, t_3], labels=["\n$t_1$", "\n$t_2$", "\n$t_3$"])
    sec.tick_params(
        'x',
        length=0,
        pad=fontsize,
        labelsize=fontsize_small
    )

    ax2.patch.set_visible(False)
    ax2.axis("off")

In [ ]:
def plot_figure_3e(ax: mpl.axes.Axes):
    cmap = plt.colormaps["viridis"]
    angle_elev = 30
    angle_rot = -20
    e_margin = 2
    psd_margin = 0.002

    df = get_figure_3e_data(e_margin, psd_margin)
    x = df["x"]
    y = df["y"]
    z = df["z"]
    dx = df["dx"]
    dy = df["dy"]
    dz = df["dz"]

    min_dz = -2000
    max_dz = np.max(dz)
    norm = mpl.colors.Normalize(vmin=min_dz, vmax=max_dz)
    mapped_colors = [cmap(norm(dz_val)) for dz_val in dz]

    ax.view_init(angle_elev, angle_rot)
    ax.bar3d(x, y, z, dx, dy, dz,
             color=mapped_colors,
             shade=False,
             zsort="max",
             lw=0.05,
             ec="black"
            )

    gamma_text = ["Gamma", "channel"]
    neutron_text = ["Neutron", "channel"]
    scaling = (0.03, 280)
    gamma_position = (4400, 0.135)
    neutron_position = (4400, 0.35)
    linespacing = 1

    for text_list, position in [
        (gamma_text, gamma_position),
        (neutron_text, neutron_position)
    ]:
        text_path = make_text_path(
            text_list,
            linespacing,
        )
        transform = mpl.transforms.Affine2D()
        transform = transform.scale(*scaling)
        pos_from = get_bbox_center(text_path.get_extents(transform))
        x_translate, y_translate = get_translation_to(pos_from, position)
        transform = transform.translate(x_translate, y_translate)
        x_center, y_center = get_bbox_center(text_path.get_extents(transform))
        transform = transform.rotate_deg_around(x_center, y_center, 90)
        text_path = transform.transform_path(text_path)
        text_patch = convert_text_path_to_patch(text_path, 1)
        patch_to_3d_plot_wall(text_patch, ax, "z")

    arrow_width = 0.04
    arrow_head_width = 0.07
    arrow_head_length = 400
    arrow_base_e = 3950
    arrow_len_psd = 0
    arrow_kwargs = {
        "width": arrow_width,
        "head_width": arrow_head_width,
        "head_length": arrow_head_length,
        "length_includes_head": True,
        "ec": "black",
        "fc": "none",
    }
    arrow_g = mpl.patches.FancyArrow(
        arrow_base_e, 0.135, -800, arrow_len_psd,
        **arrow_kwargs
    )
    patch_to_3d_plot_wall(arrow_g, ax, "z")
    arrow_n = mpl.patches.FancyArrow(
        arrow_base_e, 0.35, -2550, arrow_len_psd,
        **arrow_kwargs
    )
    patch_to_3d_plot_wall(arrow_n, ax, "z")

    ax.set_xlabel("Energy (ADC channel x1000)")
    ax.set_ylabel("PSD")
    ax.set_zlabel("Counts (x1000)")

    ax.set_xlim(0, 5000)
    ax.set_ylim(0, 0.5)
    ax.set_zlim(0, 4000)

    set_3d_axis_label_padding(ax, -1.3, -1.2, -1.5)

    ax.xaxis.set_major_formatter(lambda x, pos: f"{x / 1000:.1f}")
    ax.zaxis.set_major_formatter(lambda z, pos: f"{z / 1000:.1f}")
    ax.tick_params(labelsize=fontsize_small)
    set_3d_ticklabel_padding(ax, -0.8, -0.7, -1.0)
    set_3d_ticklabel_alignment(ax)

    set_3d_panes_transparent(ax)

### Plot Creation

In [ ]:
fig_folder = dl_folder / "figures"
fig_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
cm = 1/2.54
mm = 0.1*cm

fig_height_mm = 205
fig_height = fig_height_mm*mm
fig_width_mm = 180
fig_width = fig_width_mm*mm
print(f"Figure dimensions: {fig_width_mm} mm x {fig_height_mm} mm ({fig_width:.2f} in. x {fig_height:.2f} in.)")

row_count = 3
row_height = fig_height / row_count
heights = [row_height] * 5

col_ratios = [17, 17, 8, 9, 17]
col_total = sum(col_ratios)
col_widths = [(ratio / col_total) * fig_width for ratio in col_ratios]
width_a = sum(col_widths[:3])
width_b = sum(col_widths[3:])
width_c = sum(col_widths[:2])
width_d = sum(col_widths[2:])
width_e = sum(col_widths[1:4])
widths = [width_a, width_b, width_c, width_d, width_e]

fig_sizes = list(zip(widths, heights))
figsize_a, figsize_b, figsize_c, *rest = fig_sizes
figsize_d, figsize_e, *_ = rest

fig_names = [f"Figure {x}" for x in "ABCDE"]
for fig_name, figsize in zip(fig_names, fig_sizes):
    width, height = figsize
    print(f"{fig_name}: {width:.2f} in. wide x {height:.2f} in. tall")

In [ ]:
def save_figure(
    fig: mpl.figure.Figure,
    filename: str,
    extensions: list[str] | None = None,
    **kwargs
):
    if extensions is None:
        extensions = ["png", "eps", "pdf"]
    paths = [fig_folder / f"{filename}.{ext}" for ext in extensions]
    for path in paths:
        fig.savefig(path, **kwargs)

In [ ]:
fig, ax = plt.subplots(
    1, 1,
    figsize=figsize_a,
    dpi=600,
    layout="constrained"
)
plot_figure_3a(ax)
save_figure(fig, "fig_3a", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(
    1, 1,
    figsize=figsize_b,
    dpi=600,
    layout="constrained"
)
plot_figure_3b(ax)
save_figure(fig, "fig_3b", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(
    1, 1,
    figsize=figsize_c,
    dpi=600,
    layout="constrained"
)
plot_figure_3c(ax)
save_figure(fig, "fig_3c", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(
    1, 1,
    figsize=figsize_d,
    dpi=600,
    layout="constrained"
)
plot_figure_3d(ax)
save_figure(fig, "fig_3d", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(
    1, 1,
    figsize=figsize_e,
    dpi=600,
    subplot_kw={"projection": "3d"},
)
plot_figure_3e(ax)
ax.set_box_aspect(None, zoom=0.85)
save_figure(fig, "fig_3e", dpi=fig.dpi)

In [ ]:
input("Processing done, hit Enter to finish")
stop()

## Base Data Export

In [ ]:
excel_folder = dl_folder / "excel"
excel_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def export_figure_3a_data():
    df = get_figure_3a_data()

    column_names = df.columns.map(lambda col: f"Sample {col}")
    df.to_excel(
        excel_folder / "figure_3a.xlsx",
        header=column_names,
        index_label="Time (us)"
    )


def export_figure_3b_data():
    signals_df, histo_df = get_figure_3b_data()

    signals_df.to_excel(
        excel_folder / "figure_3b_left.xlsx",
        header=False,
        index_label=""
    )

    histo_df.to_excel(
        excel_folder / "figure_3b_right.xlsx",
        header=["Counts"],
        index_label="Light level (ADC channels)"
    )


def export_figure_3c_data():
    df = get_figure_3c_data()
    df.to_excel(
        excel_folder / "figure_3c.xlsx",
        header=["Pulse height (ADC channels)"],
        index_label="Time (ns)"
    )


def export_figure_3d_data():
    neutron_df, gamma_df = get_figure_3d_data()
    neutron_df.to_excel(
        excel_folder / "figure_3d_neutron.xlsx",
        header=["Pulse height (ADC channels)"],
        index_label="Time (ns)"
    )
    gamma_df.to_excel(
        excel_folder / "figure_3d_gamma.xlsx",
        header=["Pulse height (ADC channels)"],
        index_label="Time (ns)"
    )


def export_figure_3e_data():
    df = get_figure_3e_data()
    df.to_excel(
        excel_folder / "figure_3e.xlsx"
    )

In [ ]:
export_figure_3a_data()

In [ ]:
export_figure_3b_data()

In [ ]:
export_figure_3c_data()

In [ ]:
export_figure_3d_data()

In [ ]:
export_figure_3e_data()

In [ ]:
input("Processing done, hit Enter to finish")
stop()